# Comparison run: BERTopic on paragraphs instead of sentences

Comparison run for the choice of the unit of analysis (paper, Kap. 4.2: Satz oder Absatz). Identical to the final model in `02_bertopic` except for the documents: the **678 paragraphs** of the Lesefassung instead of the 10,039 sentences. Same embedding model, same UMAP and HDBSCAN parameters, same random seed, same stopword list.

**Embedding model**: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (the model BERTopic loads with `language="french"`; 384 dimensions, max. 128 word pieces)

**Important – input length**: the model reads at most 128 word pieces (roughly 80–90 French words). The paragraphs are much longer (median about 280 words), so with the default setting the embedding of a paragraph only represents its beginning. The cell *Embeddings* reports how many paragraphs are truncated. `EMBEDDING_STRATEGY` offers two options:
- `"truncate"` (default): the paragraph is encoded as a whole and truncated by the model – what a plain BERTopic run with the parameters of the sentence model does, i.e. comparable to the first paragraph run (Galka / Weis).
- `"mean_of_sentences"`: every sentence is encoded and the paragraph embedding is the mean of its sentence embeddings, so the whole paragraph is taken into account.

The topic representations (c-TF-IDF) are always computed from the full paragraph text.

**Input** (from `02_bertopic`)
- `../02_bertopic/input/lesefassung.txt` – plain text; paragraphs are delimited by `///`, sentences by `###`
- `../02_bertopic/stopwords/stopwords_edited.txt` – edited French stopword list, used for the topic representations only

**Output** (`output/paragraphs/`, file names contain the strategy)
- `processed_paragraphs_lesefassung.txt` – the paragraphs as fed into the model, with paragraph numbers
- `topic_info_with_weights_paragraphs_<strategy>.json` – topics with top terms and c-TF-IDF weights
- `document_info_paragraphs_<strategy>.json` – topic assignment per paragraph, with paragraph number (`Paragraph` *n* = `xml:id="P.n"`)
- `barchart_visualization_paragraphs_<strategy>.html` – bar chart of the top terms of all topics

The run is not curated, and the model is not saved.

**Environment**: as `02_bertopic` (Google Colab, Python 3.13, T4 GPU, package versions in `../02_bertopic/requirements.txt`). Run the notebook with this folder as working directory; all paths are relative.

## Setup

In [ ]:
# Install the pinned versions (uncomment if needed)
# %pip install -r ../02_bertopic/requirements.txt

# In Google Colab: mount Drive and change into the folder containing this notebook
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<path-to>/05_model_comparison

In [ ]:
import json
import os
import random
import re

import numpy as np
import torch
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# "truncate" or "mean_of_sentences", see the introduction
EMBEDDING_STRATEGY = "truncate"

OUT_DIR = "./output/paragraphs"
os.makedirs(OUT_DIR, exist_ok=True)

# Prepare data

Splits the text into paragraphs at `///` and each paragraph into sentences at `###` (the sentences are only needed for `"mean_of_sentences"`). A paragraph is the cleaned text of its sentences joined by spaces. Paragraph numbers are counted as in `02_bertopic`, so `Paragraph` *n* corresponds to `xml:id="P.n"` in the TEI.

In [ ]:
file_path = '../02_bertopic/input/lesefassung.txt'

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

def clean_text(s):
    # Replace non-breaking spaces and line breaks, collapse whitespace
    s = s.replace('\xa0', ' ').replace('\n', ' ')
    return re.sub(r'\s+', ' ', s).strip()

def process_text_for_bertopic(text):
    paragraphs, paragraph_numbers, paragraph_sentences = [], [], []
    for idx, paragraph in enumerate(text.split('///')):
        sentences = [clean_text(s) for s in paragraph.split('###') if clean_text(s)]
        if sentences:
            paragraphs.append(' '.join(sentences))
            paragraph_numbers.append(idx + 1)
            paragraph_sentences.append(sentences)
    return paragraphs, paragraph_numbers, paragraph_sentences

paragraphs, paragraph_numbers, paragraph_sentences = process_text_for_bertopic(text)

n_words = [len(p.split()) for p in paragraphs]
print(f"{len(paragraphs)} paragraphs, {sum(len(s) for s in paragraph_sentences)} sentences")
print(f"words per paragraph: median {int(np.median(n_words))}, min {min(n_words)}, max {max(n_words)}")

with open(f"{OUT_DIR}/processed_paragraphs_lesefassung.txt", 'w', encoding='utf-8') as f:
    f.write(f"The text contains {len(paragraphs)} paragraphs based on the '///' split.\n\n")
    for num, p in zip(paragraph_numbers, paragraphs):
        f.write(f"Paragraph {num}: {p}\n")

# Embeddings

Loads the same embedding model as the sentence run, reports how many paragraphs exceed its input length and computes the embeddings with the chosen strategy.

In [ ]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Truncation check: word pieces per paragraph (incl. special tokens)
max_len = embedding_model.max_seq_length
n_tokens = [len(embedding_model.tokenizer(p)["input_ids"]) for p in paragraphs]
n_truncated = sum(n > max_len for n in n_tokens)
print(f"max_seq_length: {max_len}")
print(f"word pieces per paragraph: median {int(np.median(n_tokens))}, max {max(n_tokens)}")
print(f"paragraphs truncated: {n_truncated} of {len(paragraphs)} ({n_truncated / len(paragraphs):.1%})")

if EMBEDDING_STRATEGY == "truncate":
    embeddings = embedding_model.encode(paragraphs, show_progress_bar=True)
elif EMBEDDING_STRATEGY == "mean_of_sentences":
    flat = [s for sents in paragraph_sentences for s in sents]
    sentence_embeddings = embedding_model.encode(flat, show_progress_bar=True)
    embeddings, i = [], 0
    for sents in paragraph_sentences:
        embeddings.append(sentence_embeddings[i:i + len(sents)].mean(axis=0))
        i += len(sents)
    embeddings = np.vstack(embeddings)
else:
    raise ValueError(EMBEDDING_STRATEGY)

print(embeddings.shape)

# Create topics

Same UMAP and HDBSCAN parameters as the sentence model in `02_bertopic`.

In [ ]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

topic_model = BERTopic(
    verbose=True,
    language="french",
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model
)

topics, probs = topic_model.fit_transform(paragraphs, embeddings=embeddings)

# Finetuning the model

Recomputes the topic representations with the edited stopword list. This only changes the terms describing each topic, not the assignment of paragraphs to topics.

In [ ]:
with open('../02_bertopic/stopwords/stopwords_edited.txt', 'r', encoding='utf-8') as file:
    stop_words = [word.strip() for word in file.readlines()]

vectorizer_model = CountVectorizer(stop_words=stop_words)
topic_model.update_topics(paragraphs, vectorizer_model=vectorizer_model)

# Investigate the topics

In [ ]:
topic_info_df = topic_model.get_topic_info()
n_topics = int((topic_info_df["Topic"] != -1).sum())
n_outliers = int((np.array(topics) == -1).sum())
largest = topic_info_df.loc[topic_info_df["Topic"] != -1, "Count"].max()
print(f"strategy: {EMBEDDING_STRATEGY}")
print(f"topics: {n_topics}")
print(f"paragraphs without topic: {n_outliers} of {len(paragraphs)} ({n_outliers / len(paragraphs):.1%})")
print(f"largest topic: {largest} paragraphs")
topic_info_df

In [ ]:
# Topic info with the c-TF-IDF weights of the top terms
topic_info_with_weights = []
for _, row in topic_info_df.iterrows():
    row_data = row.to_dict()
    if row["Topic"] != -1:
        row_data["Terms_with_Weights"] = topic_model.get_topic(row["Topic"])
    topic_info_with_weights.append(row_data)

output_file = f"{OUT_DIR}/topic_info_with_weights_paragraphs_{EMBEDDING_STRATEGY}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(topic_info_with_weights, f, ensure_ascii=False, indent=4, default=str)
print(f"Topic info with weights has been saved to {output_file}")

# Topic labels and document info

One entry per paragraph, with its paragraph number.

In [ ]:
topic_labels = topic_model.generate_topic_labels(nr_words=20, topic_prefix=False, separator=" - ")
topic_model.set_topic_labels(topic_labels)

document_info_df = topic_model.get_document_info(paragraphs)
document_info_df.insert(0, "Paragraph", paragraph_numbers)

output_file = f"{OUT_DIR}/document_info_paragraphs_{EMBEDDING_STRATEGY}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(document_info_df.to_json(orient='records', force_ascii=False, indent=4))
print(f"Document info has been saved to {output_file}")

# Visualize

Bar chart of the top terms of all topics.

In [ ]:
fig = topic_model.visualize_barchart(width=280, height=330, top_n_topics=n_topics, n_words=10)
fig.write_html(f"{OUT_DIR}/barchart_visualization_paragraphs_{EMBEDDING_STRATEGY}.html")
fig